# KNN K Optimizer

This notebook shows how to use `KNNKOptimizer` with a precomputed RMSD matrix. The optimizer searches for the smallest K that makes the conformer graph connected.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "knn_k_optimizer.py").is_file():
            return candidate
    raise RuntimeError("Could not find repository root")

repo = find_repo_root(Path.cwd().resolve())
src = repo / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

import numpy as np
from knn_k_optimizer import KNNKOptimizer


## Create or load an RMSD matrix

In production, load an `N x N` matrix from your conformer RMSD calculation with `np.load`. For the tutorial, this synthetic matrix has two close local clusters with a weaker bridge between them.

In [ ]:
rmsd_matrix = np.array([
    [0.0, 0.2, 0.4, 2.8, 3.0, 3.2],
    [0.2, 0.0, 0.3, 2.7, 2.9, 3.1],
    [0.4, 0.3, 0.0, 1.9, 2.1, 2.3],
    [2.8, 2.7, 1.9, 0.0, 0.2, 0.4],
    [3.0, 2.9, 2.1, 0.2, 0.0, 0.3],
    [3.2, 3.1, 2.3, 0.4, 0.3, 0.0],
])

rmsd_matrix

## Optimize K

The default `edge_mode="one_way"` adds an undirected edge if either conformer selects the other as a neighbor.

In [ ]:
optimizer = KNNKOptimizer(rmsd_matrix, k_min=1, k_max_cap=5, edge_mode="one_way")
result = optimizer.optimize()

print("optimal_k:", result.optimal_k)
print("connected:", result.connected)
print("edge_count:", result.edge_count)
print("avg_edge_weight:", round(result.avg_edge_weight, 4))

## Inspect diagnostics

Every tested K records connectivity, component count, edge count, average edge RMSD, and two scoring terms used by `strategy4_score`.

In [ ]:
for diagnostic in result.diagnostics:
    print(
        f"K={diagnostic.k}",
        f"connected={diagnostic.connected}",
        f"components={diagnostic.n_components}",
        f"edges={diagnostic.edge_count}",
        f"avg_rmsd={diagnostic.avg_edge_weight:.3f}",
        f"strategy4={optimizer.strategy4_score(diagnostic):.3f}",
    )

## Build a graph directly

Use `build_graph` when you want the edge list and neighbor table for a specific K.

In [ ]:
graph = optimizer.build_graph(result.optimal_k)

print("neighbor indices:")
print(graph.neighbor_indices)
print("edges:")
print(graph.edges)
print("edge weights:")
print(np.round(graph.edge_weights, 3))

## Mutual KNN mode

`edge_mode="mutual"` keeps only reciprocal neighbor pairs. It can require a larger K to connect the graph.

In [ ]:
mutual_optimizer = KNNKOptimizer(rmsd_matrix, k_min=1, k_max_cap=5, edge_mode="mutual")
mutual_result = mutual_optimizer.optimize()

print("mutual optimal_k:", mutual_result.optimal_k)
for diagnostic in mutual_result.diagnostics:
    print(diagnostic)